In [5]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb

def train_kidney_model():
    print("Loading dataset...")
    df = pd.read_csv('/content/kidney_disease.csv')

    # Drop id
    if 'id' in df.columns:
        df = df.drop('id', axis=1)

    # Clean up strings
    df = df.replace('\t?', np.nan)
    df = df.replace('\t', np.nan)
    df['dm'] = df['dm'].replace(to_replace={'\tno':'no', '\tyes':'yes', ' yes':'yes'})
    df['cad'] = df['cad'].replace(to_replace={'\tno':'no'})
    df['classification'] = df['classification'].replace(to_replace={'ckd\t':'ckd'})

    # Identify numerical and categorical columns
    numeric_cols = ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo']
    # pcv, wc, rc are numeric but might be strings due to \t
    for col in ['pcv', 'wc', 'rc']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_cols.append(col)

    cat_cols = ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']

    # Fill missing values
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].mean())

    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    df['classification'] = df['classification'].fillna(df['classification'].mode()[0])

    # Encode categorical features
    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le

    # Map target
    df['classification'] = df['classification'].map({'ckd': 1, 'notckd': 0})

    # Prepare features
    features = numeric_cols + cat_cols
    X = df[features]
    y = df['classification']

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled_df = pd.DataFrame(X_scaled, columns=features)

    # Train Test Split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.2, random_state=42)

    print("Training Random Forest...")
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    print("RF Score:", rf.score(X_test, y_test))

    print("Training Logistic Regression...")
    lr = LogisticRegression(random_state=42)
    lr.fit(X_train, y_train)
    print("LR Score:", lr.score(X_test, y_test))

    print("Training SVC...")
    svc = SVC(random_state=42, probability=True)
    svc.fit(X_train, y_train)
    print("SVC Score:", svc.score(X_test, y_test))

    # Save files
    base_dir = '/content/Preprocessing Files/ML-Project kidney disease prediction' # Corrected path
    os.makedirs(base_dir, exist_ok=True)

    print("Saving Preprocessing Files...")
    with open(f'{base_dir}/columns.pkl', 'wb') as f:
        pickle.dump(features, f)

    with open(f'{base_dir}/scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)

    with open(f'{base_dir}/encoder.pkl', 'wb') as f:
        pickle.dump(encoders, f)

    model_dir = '/content/Models/ML-Project-kidney-disease-Prediction-Models' # Corrected path
    os.makedirs(model_dir, exist_ok=True)

    print("Saving Models...")
    with open(f'{model_dir}/kidney_rf_model.sav', 'wb') as f:
        pickle.dump(rf, f)

    with open(f'{model_dir}/kidney_lr_model.sav', 'wb') as f:
        pickle.dump(lr, f)

    with open(f'{model_dir}/kidney_svc_model.sav', 'wb') as f:
        pickle.dump(svc, f)

    print("Done!")

if __name__ == '__main__':
    train_kidney_model()

Loading dataset...
Training Random Forest...
RF Score: 1.0
Training Logistic Regression...
LR Score: 1.0
Training SVC...
SVC Score: 1.0
Saving Preprocessing Files...
Saving Models...
Done!


## Loading and Using Saved Models for Prediction

To use the trained models for prediction on new data, you need to load the preprocessing tools (scaler and label encoders) and the models themselves. The preprocessing steps applied during training must also be applied to new data before making predictions.

In [6]:
import pickle
import pandas as pd

# Define base directories (adjust if your notebook is not in /content/)
PREPROCESSING_DIR = '/content/Preprocessing Files/ML-Project kidney disease prediction'
MODELS_DIR = '/content/Models/ML-Project-kidney-disease-Prediction-Models'

# Load preprocessing files
print("Loading preprocessing files...")
with open(f'{PREPROCESSING_DIR}/columns.pkl', 'rb') as f:
    features = pickle.load(f)

with open(f'{PREPROCESSING_DIR}/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open(f'{PREPROCESSING_DIR}/encoder.pkl', 'rb') as f:
    encoders = pickle.load(f)

print("Preprocessing files loaded successfully!")

Loading preprocessing files...
Preprocessing files loaded successfully!


In [7]:
# Load trained models
print("Loading trained models...")
with open(f'{MODELS_DIR}/kidney_rf_model.sav', 'rb') as f:
    loaded_rf_model = pickle.load(f)

with open(f'{MODELS_DIR}/kidney_lr_model.sav', 'rb') as f:
    loaded_lr_model = pickle.load(f)

with open(f'{MODELS_DIR}/kidney_svc_model.sav', 'rb') as f:
    loaded_svc_model = pickle.load(f)

print("Models loaded successfully!")
print(f"Loaded Random Forest Model: {loaded_rf_model}")
print(f"Loaded Logistic Regression Model: {loaded_lr_model}")
print(f"Loaded SVC Model: {loaded_svc_model}")

Loading trained models...
Models loaded successfully!
Loaded Random Forest Model: RandomForestClassifier(random_state=42)
Loaded Logistic Regression Model: LogisticRegression(random_state=42)
Loaded SVC Model: SVC(probability=True, random_state=42)


### Making a Prediction on New Data

Now, let's simulate a new patient's data and predict whether they have kidney disease. Remember to apply the same preprocessing steps (encoding, scaling) as during training.